In [1]:
import pandas as pd

# Define file paths
l89_train_path = '/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/data_dir_l89_L2SR/l89_temporal_16_resized_to_224_CRSfixed/train_2025_balanced.csv'
l89_test_path = '/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/data_dir_l89_L2SR/l89_temporal_16_resized_to_224_CRSfixed/test_2025_balanced.csv'
s2_train_path = '/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/s2_90360_temporal_CDSE0_gee90360_2024_16/train.csv'
s2_test_path = '/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/s2_90360_temporal_CDSE0_gee90360_2024_16/test.csv'

def process_and_merge(l89_file, s2_file, output_name):
    # Load datasets
    df_l89 = pd.read_csv(l89_file)
    df_s2 = pd.read_csv(s2_file)

    # 1. Add sensor labels
    df_l89['sensor'] = 'l89'
    df_s2['sensor'] = 's2'

    # 2. Align S2 columns to match L89 structure
    # Mapping s2 columns to the requested "path_t0, path_t90, path_t360"
    s2_rename_map = {
        'image_path': 'path_t0',
        's2_pre_path': 'path_t90',
        's2_pre_pre_path': 'path_t360'
    }
    df_s2 = df_s2.rename(columns=s2_rename_map)

    # 3. Keep only the shared/relevant columns
    common_cols = ['id', 'label', 'path_t0', 'path_t90', 'path_t360', 'latitude', 'longitude', 'datetime', 'sensor']
    
    # Ensure columns exist in both (L89 already has them, S2 now has them via rename)
    df_l89_subset = df_l89[common_cols]
    df_s2_subset = df_s2[common_cols]

    # 4. Concatenate and shuffle
    combined_df = pd.concat([df_l89_subset, df_s2_subset], axis=0)
    combined_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)

    # 5. Save to CSV
    combined_df.to_csv(output_name, index=False)
    print(f"Saved merged file: {output_name} | Total rows: {len(combined_df)}")

# Run for Train and Test sets
process_and_merge(l89_train_path, s2_train_path, '/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/s2_90360_temporal_CDSE0_gee90360_2024_16/merged_s2l89_train.csv')
process_and_merge(l89_test_path, s2_test_path, '/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/s2_90360_temporal_CDSE0_gee90360_2024_16/merged_s2l89_test.csv')

Saved merged file: /mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/s2_90360_temporal_CDSE0_gee90360_2024_16/merged_s2l89_train.csv | Total rows: 91511
Saved merged file: /mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/s2_90360_temporal_CDSE0_gee90360_2024_16/merged_s2l89_test.csv | Total rows: 26145
